# Blif to Graph

This code is a simple parser for the BLIF format. It reads a BLIF file and creates a graph with the gates and connections between them. This graph is used to generate a cuda kernel that evaluates the circuit.

In [49]:
import os
import networkx as nx
import numpy as np
import math

In [50]:
N    = 4
PATH = f'{os.getcwd()}/results/blif_generator'
# Verify if the directory exists
if not os.path.exists(PATH):
    os.makedirs(PATH)



print(PATH)

/app/results/blif_generator


## BLIF parser

In [51]:
import networkx as nx
from itertools import product

def binary_to_decimal(binary_str):
    return int(binary_str, 2)


def pla_to_binary_vector(pla_table, num_vars):
    """
    Converte uma tabela PLA para um vetor binário na ordem correta.
    """
    vector_size = 2 ** num_vars
    binary_vector = [0 if int(pla_table[0]['output'])==1 else 1] * vector_size

    
    
    for entry in pla_table:
        pattern, output = ''.join(entry["input"]), entry["output"]  # Garante que input seja tratado como string
        positions = [i for i, char in enumerate(pattern) if char == '-']
        base_pattern = list(pattern)
        
        for replacements in product('01', repeat=len(positions)):
            temp_pattern = base_pattern[:]
            for pos, char in zip(positions, replacements):
                temp_pattern[pos] = char
            expanded_pattern = ''.join(temp_pattern)
            try:
                index = int(expanded_pattern, 2)  # Garante que a ordem dos bits seja respeitada
                binary_vector[index] = int(output)  # Armazena o valor correto
            except ValueError:
                print(f"Erro ao converter: {expanded_pattern}")
    
    binary_vector.reverse()
    return binary_vector

def parse_blif(file_path):
    G = nx.DiGraph()
    
    with open(file_path, 'r') as file:
        lines = file.readlines()
    
    inputs, outputs = [], []
    parsing_names = False
    current_node = None
    node_data = {}
    input_order = {}
    
    for line in lines:
        tokens = line.strip().split()

        if tokens[0] == ".inputs":
            inputs = tokens[1:]
            G.add_nodes_from(inputs, type="input")
        elif tokens[0] == ".outputs":
            outputs = tokens[1:]
            G.add_nodes_from(outputs, type="output")
        elif tokens[0] == ".names":
            parsing_names = True
            current_node = tokens[-1]
            dependencies = tokens[1:-1]
     
            node_data[current_node] = []
   
            input_order[current_node] = dependencies

            G.nodes[current_node]['input_order'] = dependencies
            G.nodes[current_node]['truth_value'] = None

            for dep in dependencies:
                if dep not in G.nodes():
                    G.add_node(dep, type="intermediary", input_order=[], truth_value=None)
                G.add_edge(dep, current_node)
        elif parsing_names and current_node is not None and len(tokens) > 1:
            input_values = tokens[:-1]
            output_value = tokens[-1]
            node_data[current_node].append({"input": input_values, "output": output_value})
    
    for node, partial_truth_table in node_data.items():
        num_vars = len(input_order[node])
        print(f"Node {node} has partial truth table: {partial_truth_table}")
        full_truth_table = pla_to_binary_vector(partial_truth_table, num_vars)
        print(f"Node {node} has truth table: {full_truth_table}")
        G.nodes[node]['truth_value'] = binary_to_decimal("".join(map(str, full_truth_table)))
    
    return G

# Exemplo de uso
graph = parse_blif(f"{PATH}/somador.blif")
graph.nodes(data=True)


Node z0 has partial truth table: [{'input': ['---01'], 'output': '0'}, {'input': ['000-1'], 'output': '0'}]
Node z0 has truth table: [1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1]
Node new_n16 has partial truth table: [{'input': ['--011'], 'output': '1'}, {'input': ['01--1'], 'output': '1'}]
Node new_n16 has truth table: [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0]
Node new_n17 has partial truth table: [{'input': ['-00-0'], 'output': '1'}, {'input': ['00-00'], 'output': '1'}, {'input': ['00111'], 'output': '1'}, {'input': ['01110'], 'output': '1'}, {'input': ['1-010'], 'output': '1'}]
Node new_n17 has truth table: [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1]
Node new_n18 has partial truth table: [{'input': ['-0001'], 'output': '0'}, {'input': ['-011-'], 'output': '0'}, {'input': ['00--1'], 'output': '0'}, {'input': ['1-010'], 'output'

NodeDataView({'x0': {'type': 'input'}, 'x1': {'type': 'input'}, 'x2': {'type': 'input'}, 'x3': {'type': 'input'}, 'x4': {'type': 'input'}, 'x5': {'type': 'input'}, 'x6': {'type': 'input'}, 'x7': {'type': 'input'}, 'z0': {'type': 'output', 'input_order': ['new_n22', 'new_n16', 'new_n19', 'x3', 'new_n25'], 'truth_value': 3722304981}, 'z1': {'type': 'output', 'input_order': ['new_n29', 'new_n20', 'new_n31', 'new_n33'], 'truth_value': 21844}, 'z2': {'type': 'output', 'input_order': ['new_n35', 'x1', 'x5'], 'truth_value': 105}, 'z3': {'type': 'output', 'input_order': ['new_n30', 'x2', 'x6'], 'truth_value': 150}, 'z4': {'type': 'output', 'input_order': ['x3', 'x7'], 'truth_value': 6}, 'new_n22': {'type': 'intermediary', 'input_order': ['x1', 'x7', 'new_n24', 'new_n23', 'x5'], 'truth_value': 1079002196}, 'new_n16': {'type': 'intermediary', 'input_order': ['new_n17', 'x4', 'new_n18', 'x0', 'x5'], 'truth_value': 134785544}, 'new_n19': {'type': 'intermediary', 'input_order': ['new_n21', 'new_n20

## CUDA kernel generation

In [ ]:
import os
import networkx as nx

def generate_cuda_code(G):
    # Count inputs and outputs
    inputs = [node for node, data in G.nodes(data=True) if data.get('type') == 'input']
    outputs = [node for node, data in G.nodes(data=True) if data.get('type') == 'output']
    num_inputs = len(inputs)
    num_outputs = len(outputs)
    
    # Get nodes in topological order
    ordered_nodes = list(nx.topological_sort(G))
    
    cuda_code = """
#include <stdio.h>
#include <cuda_runtime.h>
#include <stdlib.h>

// Utility function to check CUDA errors
#define CHECK_CUDA_ERROR(call) { \\
    cudaError_t err = call; \\
    if (err != cudaSuccess) { \\
        printf("CUDA error: %s\\n", cudaGetErrorString(err)); \\
        exit(1); \\
    } \\
}

__global__ void evaluate_circuit(int *output) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int num_combinations = 1 << """ + str(num_inputs) + """;
    
    if (idx < num_combinations) {
        int x = idx;
"""
    
    # Initialize all variables to 0
    all_nodes = set(G.nodes())
    cuda_code += "\n        // Initialize all variables\n"
    for node in all_nodes:
        cuda_code += f"    unsigned int {node};\n"
    
    cuda_code += "\n        // Set input values\n"
    # Input variable assignments
    for i, input_var in enumerate(reversed(inputs)):
        cuda_code += f"        {input_var} = (x >> {i}) & 1;\n"
    
    cuda_code += "\n        // Evaluate circuit in topological order\n"
    
    # Generate equations in topological order
    for node in ordered_nodes:
        if node not in inputs:  # Skip input nodes
            input_order = G.nodes[node].get('input_order', [])
            truth_value = G.nodes[node].get('truth_value')
            
            if truth_value is not None and input_order:
                shifts = []
                for i, input_var in enumerate(input_order):
                    shifts.append(f"({input_var} << {len(input_order) - 1 - i})")
                equation = f"        {node} = (({truth_value}U) >> (" + " + ".join(shifts) + ")) & 1;\n"
                cuda_code += equation
    
    cuda_code += "\n        // Combine outputs into a single integer\n"
    cuda_code += "        output[idx] = 0;\n"
    
    # Combine outputs
    for i, output in enumerate(reversed(outputs)):
        cuda_code += f"        output[idx] |= ({output} << {i});\n"

    
    cuda_code += "    }\n}\n\n"
    
    # Add main function with timing and validation
    cuda_code += """
int main() {
    // Calculate number of input combinations
    const int num_inputs = """ + str(num_inputs) + """;
    const int num_combinations = 1 << num_inputs;
    const int output_size = num_combinations * sizeof(int);
    
    // Create CUDA events for timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    
    // Allocate host memory
    int *h_output = (int*)malloc(output_size);
    
    // Allocate device memory
    int *d_output;
    CHECK_CUDA_ERROR(cudaMalloc(&d_output, output_size));
    
    // Calculate grid and block dimensions
    int threadsPerBlock = 1024;
    // int blocksPerGrid = (num_combinations + threadsPerBlock - 1) / threadsPerBlock;
    int blocksPerGrid  = 1024*1024;
    
    // Record start time
    cudaEventRecord(start);
    
    // Launch kernel
    evaluate_circuit<<<blocksPerGrid, threadsPerBlock>>>(d_output);
    CHECK_CUDA_ERROR(cudaGetLastError());
    
    // Record stop time
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    
    // Calculate elapsed time
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    
    // Copy result back to host
    CHECK_CUDA_ERROR(cudaMemcpy(h_output, d_output, output_size, cudaMemcpyDeviceToHost));
    
    // Print timing information
    printf("\\nGPU Execution time: %.3f milliseconds\\n", milliseconds);
    
    // Print header only if we find invalid results
    bool header_printed = false;
    
    // Check results and print only invalid ones
    for (int i = 0; i < num_combinations; i++) {
        // Get input values
        int input1 = 0;
        int input2 = 0;
        
        // Extract first half of inputs for input1
        for (int j = num_inputs/2 - 1; j >= 0; j--) {
            input1 = (input1 << 1) | ((i >> (j + num_inputs/2)) & 1);
        }
        
        // Extract second half of inputs for input2
        for (int j = num_inputs/2 - 1; j >= 0; j--) {
            input2 = (input2 << 1) | ((i >> j) & 1);
        }
        
        // Calculate expected sum
        int expected_sum = input1 + input2;
        
        // Get actual output
        int actual_output = h_output[i];
        
        // Only print if the result is invalid
        if (actual_output != expected_sum) {
            // Print header if this is the first invalid result
            if (!header_printed) {
                printf("\\nInvalid Results:\\n");
                printf("Inputs");
                for (int k = 0; k < """ + str(num_inputs//3) + """; k++) {
                    printf("\\t");
                }
                printf("-> Output | Expected\\n");
                header_printed = true;
            }
            
            // Print input combinations
            for (int j = num_inputs - 1; j >= 0; j--) {
                printf("%d ", (i >> j) & 1);
            }
            printf("\\t-> ");
            
            // Print output values
            for (int j = """ + str(num_outputs - 1) + """; j >= 0; j--) {
                printf("%d", (actual_output >> j) & 1);
            }
            
            printf(" | Expected: %d, Got: %d\\n", expected_sum, actual_output);
        }
    }
    
    // Print message if no invalid results were found
    if (!header_printed) {
        printf("\\nAll results are valid!\\n");
    }
    
    // Cleanup
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    free(h_output);
    cudaFree(d_output);
    
    return 0;
}
"""
    return cuda_code

def save_cuda_code(cuda_code, filename=f"{PATH}/circuit_evaluator.cu"):
    """Saves the CUDA code to a file."""
    with open(filename, "w") as f:
        f.write(cuda_code)
    print(f"CUDA code saved to {filename}")

# Example usage:
cuda_code = generate_cuda_code(graph)
if cuda_code:
    save_cuda_code(cuda_code)
    print("\nTo compile and run:")
    print("1. nvcc circuit_evaluator.cu -o circuit_evaluator")
    print("2. ./circuit_evaluator")

CUDA code saved to /app/results/blif_generator/circuit_evaluator.cu

To compile and run:
1. nvcc circuit_evaluator.cu -o circuit_evaluator
2. ./circuit_evaluator


In [58]:
# Run CUDA code
os.system(f"nvcc {PATH}/circuit_evaluator.cu -o {PATH}/circuit_evaluator")
os.system(f"ncu {PATH}/circuit_evaluator")

==PROF== Connected to process 10830 (/app/results/blif_generator/circuit_evaluator)
==PROF== Profiling "evaluate_circuit(int *)" - 0: 0%....50%....100% - 8 passes

GPU Execution time: 495.087 milliseconds

Invalid Results:
Inputs		-> Output | Expected
0 0 1 1 1 1 0 1 	-> 00000 | Expected: 16, Got: 0
0 0 1 1 1 1 1 1 	-> 00010 | Expected: 18, Got: 2
0 1 0 1 1 1 0 0 	-> 00001 | Expected: 17, Got: 1
0 1 0 1 1 1 0 1 	-> 00010 | Expected: 18, Got: 2
0 1 0 1 1 1 1 0 	-> 00011 | Expected: 19, Got: 3
0 1 1 1 1 1 0 0 	-> 00011 | Expected: 19, Got: 3
0 1 1 1 1 1 0 1 	-> 00100 | Expected: 20, Got: 4
0 1 1 1 1 1 1 1 	-> 00110 | Expected: 22, Got: 6
1 0 0 1 1 1 0 0 	-> 00101 | Expected: 21, Got: 5
1 0 0 1 1 1 0 1 	-> 00110 | Expected: 22, Got: 6
1 0 0 1 1 1 1 0 	-> 00111 | Expected: 23, Got: 7
1 0 1 1 1 1 0 0 	-> 00111 | Expected: 23, Got: 7
1 0 1 1 1 1 0 1 	-> 01000 | Expected: 24, Got: 8
1 0 1 1 1 1 1 0 	-> 01001 | Expected: 25, Got: 9
1 0 1 1 1 1 1 1 	-> 01010 | Expected: 26, Got: 10
1 1 0 1 1 1 

0

In [1]:
x = 0b00001
cin = (x >> 0) & 1;
b01 = (x >> 1) & 1;
b00 = (x >> 2) & 1;
a01 = (x >> 3) & 1;
a00 = (x >> 4) & 1;

print(cin, b01, b00, a01, a00)

1 0 0 0 0
